# Run a training script as a command job

You can use the Python SDK for Azure Machine Learning to submit scripts as command jobs. By using jobs, you can easily keep track of the input parameters and outputs when training a machine learning model.

## Before you start

You'll need the latest version of the  **azure-ai-ml** package to run the code in this notebook. Run the cell below to verify that it is installed.

> **Note**:
> If the **azure-ai-ml** package is not installed, run `pip install azure-ai-ml` to install it.

In [ ]:
pip show azure-ai-ml

## Connect to your workspace

With the required SDK packages installed, now you're ready to connect to your workspace.

To connect to a workspace, we need identifier parameters - a subscription ID, resource group name, and workspace name. Since you're working with a compute instance, managed by Azure Machine Learning, you can use the default values to connect to the workspace.

In [3]:
import sys
import os
from pathlib import Path


notebook_loc = os.getcwd()
# Path to your project root
project_dir = Path(r"C:\Users\dmika\DEV\Projects-local\dp100-learn")

# Add to sys.path if not already there
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))
# Now you can import from utils
from utils.consts import SUBSCRIPTION_ID, PREFERED_RESOURCE_LOCATION, MAIN_STORAGE_ACCOUNT_ACCESS_KEY
from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient


subscription_id = SUBSCRIPTION_ID
azure_credentials = DefaultAzureCredential()

resource_group_name = "ml-workspace-dev"
resource_group_location = PREFERED_RESOURCE_LOCATION
azureml_workspace_name = "mlw-dp100-labs"
azureml_resource_location = PREFERED_RESOURCE_LOCATION
ml_client = MLClient(
    credential=azure_credentials,
    subscription_id=subscription_id,
    resource_group_name=resource_group_name,
    workspace_name=azureml_workspace_name
)
os.chdir(notebook_loc)

In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    # Check if given credential can get token successfully.
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    # Fall back to InteractiveBrowserCredential in case DefaultAzureCredential not work
    credential = InteractiveBrowserCredential()

In [ ]:
# Get a handle to workspace
ml_client = MLClient.from_config(credential=credential)

## Initiate a command job

Run the cell below to train a classification model to predict diabetes. The model is trained by running the **train-model-parameters.py** script that can be found in the **src** folder. It uses the **diabetes.csv** file as the training data. 

- `code`: specifies the folder that includes the script to run.
- `command`: specifies what to run exactly.
- `environment`: specifies the necessary packages to be installed on the compute before running the command.
- `compute`: specifies the compute to use to run the command.
- `display_name`: the name of the individual job.
- `experiment_name`: the name of the experiment the job belongs to.

Note that the command used to test the script in the terminal is the same as the command in the configuration of the job below. 

In [4]:
from azure.ai.ml import command

# configure job

job = command(
    code="./src",
    command="python train-model-parameters.py --training_data diabetes.csv",
    # environment="AzureML-sklearn-0.24-ubuntu18.04-py37-cpu@latest",
    environment="dmdp100env@latest",
    compute="dmdp100-cpu-cluster",
    display_name="diabetes-train-script",
    experiment_name="diabetes-training"
    )

# submit job
returned_job = ml_client.create_or_update(job)
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading src (0.53 MBs): 100%|#######

Monitor your job at https://ml.azure.com/runs/mango_kitchen_nt2kpl305b?wsid=/subscriptions/a1267753-4c98-48c1-a8e9-9c7169202ffd/resourcegroups/ml-workspace-dev/workspaces/mlw-dp100-labs&tid=50c76291-0c80-4444-a2fb-4f8ab168c311
